## Practical 1 — From Real Biological Data to an ML-Ready Dataset
## Bladder cancer microarray data, data quality, grouping, and batch effects

## Learning goals

By the end of this practical, you should be able to:

- identify observations, features, metadata, targets, and technical variables in a real biological dataset;
- recognize the orientation and dimensionality of an expression matrix;
- apply tidy-data principles to biological measurements and metadata;
- perform systematic data diagnostics even when no obvious data-entry errors are present;
- transform a real expression dataset into an ML-ready representation;
- explain why repeated or highly related observations must remain in the same data split;
- distinguish **real batch structure** from an **artificial pseudo grouping**;
- detect evidence of batch effects and target–batch confounding;
- independently apply the same workflow to the UCI Mice Protein Expression dataset.

**Main idea:** a model sees numerical representations. It does not assume which variation is biological, technical, duplicated, or leaked across the train/test boundary.

# 0. Dataset check


We use the Bioconductor **`bladderbatch`** dataset: an Affymetrix microarray bladder-cancer dataset containing **57 samples, 22,283 probe-set expression measurements, and five processing batches**.

The Bioconductor metadata include:

- `sample` — sample number;
- `outcome` — original biological outcome;
- `batch` — processing batch defined by the date on which arrays were processed;
- `cancer` — a simplified biological outcome.

The expression data were RMA-normalized before inclusion in the package.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer

DATA = Path(".")
RANDOM_STATE = 100926

In [ ]:
expression_raw = pd.read_csv(DATA / "bladder_expression_raw.csv")
metadata = pd.read_csv(DATA / "bladder_metadata.csv")

print("Raw expression:", expression_raw.shape)
print("Metadata:", metadata.shape)

In [ ]:
metadata

In [ ]:
expression_raw.head()

## 1.2 Prepare the expression matrix for scikit-learn

The raw export preserves the original assay orientation: probe sets are rows and biological samples are columns. Most scikit-learn workflows/tidy data principles expect the opposite orientation: samples are rows and features are columns.

We now transpose the matrix in Python and use the sample identifiers from the column names as `sample_id`. The resulting `expression` table is the ML-ready expression table used below.

In [ ]:
expression = expression_raw.set_index("probe_set_id").T.rename_axis("sample_id").reset_index()
expression.columns.name = None

print(expression_raw.shape)
print(expression.shape)


In [ ]:
expression

### First questions

Before doing any transformation, answer:

1. What is one **biological observation**?
2. What is a **feature**?
3. Which table contains biological measurements?
4. Which table contains experimental metadata?
5. Which column could be a prediction target?
6. Which column describes a known technical source of variation?

## 1.3 The dimensionality problem

In [ ]:
n_samples = expression.shape[0]
n_features = expression.shape[1] - 1

print("Samples:", n_samples)
print("Expression features:", n_features)
print("p / n =", round(n_features / n_samples, 1))

In this dataset there are tens of thousands of measured variables but only dozens of biological samples.

This is extremely common in transcriptomics and other omics datasets.

> We are **not** solving the high-dimensional modelling problem today.

# 2. EDA and tidy-data principles

A useful working definition of tidy data is:

> **One observation per row, one variable per column, one value per cell.**


## 2.1 Inspect metadata

In [ ]:
metadata.info()

## 2.2 Inspect expression values

In [ ]:
gene_cols = [c for c in expression.columns if c != "sample_id"]

expression[gene_cols].describe().T.head(10)

In [ ]:
expression[gene_cols[0]].hist(bins=20)
plt.xlabel("RMA-normalized expression")
plt.ylabel("Number of samples")
plt.title(f"Distribution of {gene_cols[0]}")
plt.show()

## 2.3 Biological outcome structure

In [ ]:
metadata["outcome"].value_counts().plot(kind="bar")
plt.ylabel("Number of samples")
plt.title("Original biological outcomes")
plt.xticks(rotation=45, ha="right")
plt.show()

In [ ]:
metadata["cancer"].value_counts().plot(kind="bar")
plt.ylabel("Number of samples")
plt.title("Simplified cancer grouping")
plt.xticks(rotation=0)
plt.show()

### Discussion

Why might the original `outcome` and simplified `cancer` variables be useful for different questions?

# 3. Data diagnostics

A diagnostic workflow does **not** mean that we must discover corrupted data.

Real datasets should be checked systematically even when they appear clean.

## 3.1 Are sample identifiers unique?

In [ ]:
print("Expression IDs unique:", expression["sample_id"].is_unique)
print("Metadata IDs unique:", metadata["sample_id"].is_unique)

## 3.2 Do both tables contain the same samples?

In [ ]:
expr_ids = set(expression["sample_id"])
meta_ids = set(metadata["sample_id"])

print("Expression only:", expr_ids - meta_ids)
print("Metadata only:", meta_ids - expr_ids)

## 3.4 Exact duplicate samples

In [ ]:
print("Duplicate metadata rows:", metadata.duplicated().sum())
print("Duplicate expression rows:", expression.duplicated().sum())

We can also ask whether any two expression rows are numerically identical.

In [ ]:
expr_only = expression.set_index("sample_id")
print("Duplicated expression profiles:",
      expr_only.duplicated().sum())

## 3.5 Constant and low-variance features

In [ ]:
variances = expression[gene_cols].var()

print("Constant features:", (variances == 0).sum())
print("Median feature variance:", variances.median())

In [ ]:
variances.hist(bins=50)
plt.xlabel("Variance across samples")
plt.ylabel("Number of probe sets")
plt.title("Distribution of expression-feature variance")
plt.show()


# 4. Data preparation

## 4.1 Merge measurements and metadata

Because both tables should have one row per sample, we can enforce a one-to-one relationship.

In [ ]:
df_real = metadata.merge(
    expression,
    on="sample_id",
    validate="one_to_one")

print(df_real.shape)
df_real.iloc[:5, :10]



## 4.2 Define a possible target

For demonstration, we will use the simplified `cancer` variable as the candidate target.

In [ ]:
y_real = df_real["cancer"].copy()
y_real.value_counts()

## 4.3 Define the real feature matrix

In [ ]:
X_real = df_real[gene_cols].copy()

print("X:", X_real.shape)
print("y:", y_real.shape)

Notice what is **not** in `X_real`:

- `sample_id`;
- `sample`;
- `outcome`;
- `batch`;
- `cancer`.

Some are identifiers, one is our target, and others are metadata whose inclusion would change the scientific question.




# 5. Artificial within-group similarity for the splitting exercise

The original bladderbatch samples are real biological samples, but the dataset does not provide the repeated-subject structure we want for this exercise. 

We will therefore construct **one pseudo-replicate per real sample** to intentioanlly spoil perfectly good dataset;)

For each real expression profile x_i, we create:

x_i_p = x_i + epsilon

epsilon - small noise

The pseudo-replicate is therefore **highly similar** to its source sample.

## 5.1 Generate pseudo-replicates

In [ ]:
rng = np.random.default_rng(RANDOM_STATE)

real = df_real.copy()
real["source_sample_id"] = real["sample_id"]
real["p_group"] = "TG_" + real["sample_id"].astype(str)


pseudo = df_real.copy()
pseudo["source_sample_id"] = pseudo["sample_id"]
pseudo["p_group"] = "TG_" + pseudo["sample_id"].astype(str)


# Give pseudo-replicates unique row/sample identifiers.
pseudo["sample_id"] = pseudo["sample_id"].astype(str) + "_pseudo"

# Small noise relative to the scale ofexpression.
noise_sigma = 0.25
noise = rng.normal(loc=0, scale=noise_sigma, size=(len(pseudo), len(gene_cols)))

pseudo.loc[:, gene_cols] = (pseudo[gene_cols].to_numpy() + noise)

df = pd.concat([real, pseudo], ignore_index=True)

print("Original samples:", len(real))
print("Pseudo-replicates:", len(pseudo))
print("pseudo rows:", len(df))
print("pseudo groups:", df["p_group"].nunique())

### Important provenance rule

In [ ]:
df[["sample_id", "source_sample_id", "p_group",  "cancer", "batch"]].head(100)

`source_sample_id`, and `p_group` are synthetic metadata.

The original `batch`, `outcome`, and `cancer` values remain those of the real sample.

## 5.2 Verify within-group similarity

Calculate the correlation between every real sample and its pseudo-replicate.

In [ ]:
pair_correlations = []

for group, group_df in df.groupby("p_group"):
    if len(group_df) != 2:
        continue

    a = group_df.iloc[0][gene_cols].to_numpy(dtype=float)
    b = group_df.iloc[1][gene_cols].to_numpy(dtype=float)

    pair_correlations.append(np.corrcoef(a, b)[0, 1])

pair_correlations = pd.Series(pair_correlations)

pair_correlations

In [ ]:
pair_correlations.hist(bins=20)
plt.xlabel("Real–pseudo expression correlation")
plt.ylabel("Number of p groups")
plt.title("Artificial within-group similarity")
plt.show()

## 5.3 Introduce synthetic missing values (pseudoNA)
To spoil our good dataset even more we will intoduce some NA.
We alter expression measurements only; sample identifiers, metadata, targets, and grouping variables remain available for the exercise.

In [ ]:
pseudo_na_rng = np.random.default_rng(RANDOM_STATE + 1)

df_pseudona = df.copy()
expression_values = df_pseudona[gene_cols].to_numpy(dtype=float).copy()

# Remove 0.01% of expression cells

pseudo_na_fraction = 0.0001
n_expression_cells = expression_values.size
n_pseudona = max(1, int(round(n_expression_cells * pseudo_na_fraction)))
flat_indices = pseudo_na_rng.choice(
    n_expression_cells,
    size=n_pseudona,
    replace=False
)
rows, columns = np.unravel_index(
    flat_indices,
    expression_values.shape
)
expression_values[rows, columns] = np.nan
df_pseudona.loc[:, gene_cols] = expression_values

print("pseudo rows:", len(df_pseudona))
print("Synthetic missing expression cells:", df_pseudona[gene_cols].isna().sum().sum())

## 5.4 Missing-value handling: detection, removal, and imputation

Missing values should be handled as part of the data-generating and modelling workflow. Start by detecting where they occur and how much information is affected.

There is no universally correct response:

- **Detection:** inspect missingness by table, row, column, and experimental group.
- **Removal:** remove rows or features only when the loss is acceptable and the rule is scientifically justified. In high-dimensional data, dropping every row with any missing feature can discard most observations.
- **Imputation:** replace missing values using a rule such as a median, mean, K-nearest neighbours, or a model-based method. The imputation rule must be fitted on the training data only, then applied to the test data, so that test-set information does not leak into training.

The following cells are a demonstration using the synthetic pseudoNA copy.

In [ ]:
missing_by_column = df_pseudona[gene_cols].isna().sum()
missing_by_row = df_pseudona[gene_cols].isna().sum(axis=1)

print("Total missing expression cells:", int(missing_by_column.sum()))
print("Features with at least one missing value:", int((missing_by_column > 0).sum()))
print("Rows with at least one missing value:", int((missing_by_row > 0).sum()))

display(missing_by_column[missing_by_column > 0].head())
missing_by_row.describe()

### Removal as a possible strategy

Complete-row removal is easy to write, but it can be very costly. Feature removal is another option, although removing a feature because of one missing value may also discard useful measurements. Compare both consequences before choosing a rule.

In [ ]:
df_rows_removed = df_pseudona.dropna(subset=gene_cols)
features_without_missing = [
    column for column in gene_cols
    if df_pseudona[column].notna().all()
]
X_features_removed = df_pseudona[features_without_missing].copy()

print("Rows before removal:", len(df_pseudona))
print("Rows after complete-row removal:", len(df_rows_removed))
print("Features before removal:", len(gene_cols))
print("Features after removal of any incomplete feature:", X_features_removed.shape[1])

### Imputation as a possible strategy

Here we use median imputation as a simple baseline. In a real analysis, the choice should reflect the measurement process and the scientific question. 

In [ ]:

X_pseudona = df_pseudona[gene_cols].copy()

imputer = SimpleImputer(strategy="median")
X_imputed = pd.DataFrame(imputer.fit_transform(X_pseudona), columns=gene_cols)


X_imputed.isna().sum().sum()

# 6. Splits

## 6.1 Naive random row-wise split

In [ ]:
row_indices = np.arange(len(df))

train_idx_random, test_idx_random = train_test_split(
    row_indices,
    test_size=0.25,
    random_state=RANDOM_STATE,
    stratify=df["cancer"])

random_train = df.iloc[train_idx_random]
random_test = df.iloc[test_idx_random]

Lets check if some pseudo groups is present in both sets

In [ ]:
random_train_groups = set(random_train["p_group"])
random_test_groups = set(random_test["p_group"])

leaked_groups = random_train_groups & random_test_groups

print("Groups appearing in both train and test", len(leaked_groups))

list(sorted(leaked_groups))

These rows are not independent. A pseudo-replicate in the test set may be almost identical to its source sample in training.


## 6.2 Group-aware split

In [ ]:
splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.25,
    random_state=RANDOM_STATE)

train_idx, test_idx = next(
    splitter.split(
        df[gene_cols],
        df["cancer"],
        groups=df["p_group"]))

train_df = df.iloc[train_idx].copy()
test_df = df.iloc[test_idx].copy()

In [ ]:
train_groups = set(train_df["p_group"])
test_groups = set(test_df["p_group"])

print("Group overlap", train_groups & test_groups)

Expected:

```text
set()
```

## 6.3 Verify that each original/pseudo pair stays together

In [ ]:
split_membership = pd.Series("test", index=df.index)
split_membership.iloc[train_idx] = "train"

check = (
    df.assign(split=split_membership)
      .groupby("p_group")["split"]
      .nunique())

print("Groups split across partitions",(check > 1).sum())

### Discussion

For the artificial pseud data, `p_group` plays the role that variables such as these often play in real studies:

- patient ID;
- donor ID;
- animal ID;
- family;
- experimental unit;
- repeated-measure subject;
- original image before augmentation;
- original molecule before generating conformers.

# 7. Real batch effects and confounding

Now we return to a property that is **not synthetic**.

`batch` is real metadata from the bladderbatch experiment and reflects the dates on which microarrays were processed.

## 7.1 Examine the experimental design

Use **only the real samples** here so that our pseudo-replicates do not alter counts.

In [ ]:
pd.crosstab(real["batch"], real["cancer"],margins=True)

In [ ]:
pd.crosstab(real["batch"], real["cancer"], normalize="index").round(2)

### Question

Is biological class independent of processing batch?

## 7.2 Original outcomes by batch

In [ ]:
outcome_batch = pd.crosstab(real["batch"], real["outcome"])

outcome_batch

In [ ]:
outcome_batch.plot(kind="bar", stacked=True)

plt.xlabel("Processing batch")
plt.ylabel("Number of real samples")
plt.title("Biological outcome composition differs across batches")
plt.legend(title="Outcome", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

# 8. Visualizing batch structure

We will use PCA only as a **diagnostic visualization** here.

The mathematical details of PCA will be covered in a later practical.

## 8.1 Select highly variable features for visualization

Using all 22,283 features is unnecessary for a quick exploratory plot. So we will keep only top 1000 by variation

In [ ]:
real_X = real[gene_cols]

feature_variance = real_X.var().sort_values(ascending=False)
top_genes = feature_variance.head(1000).index

X_vis = real_X[top_genes]

## 8.2 Standardize and project to two dimensions

In [ ]:
X_scaled = StandardScaler().fit_transform(X_vis)

pca = PCA(n_components=2)
coords = pca.fit_transform(X_scaled)

pca_df = pd.DataFrame({
    "PC1": coords[:, 0],
    "PC2": coords[:, 1],
    "batch": real["batch"].astype(str).values,
    "cancer": real["cancer"].astype(str).values,
    "outcome": real["outcome"].astype(str).values
})

print(
    "Explained variance:",
    pca.explained_variance_ratio_.round(3)
)

## 8.3 Color by batch

In [ ]:
for batch, group in pca_df.groupby("batch"):
    plt.scatter(
        group["PC1"],
        group["PC2"],
        label=f"Batch {batch}"
    )

plt.xlabel("PC1")
plt.ylabel("PC2")
plt.title("PCA of real bladderbatch samples — colored by batch")
plt.legend()
plt.show()

## 8.4 Color by cancer class

In [ ]:
for label, group in pca_df.groupby("cancer"):
    plt.scatter(
        group["PC1"],
        group["PC2"],
        label=label
    )

plt.xlabel("PC1")
plt.ylabel("PC2")
plt.title("The same samples — colored by biological class")
plt.legend()
plt.show()

### Discussion

Look at the **same geometric arrangement** twice:

1. colored by processing batch;
2. colored by biological target.

If batch and target are associated, how confidently can we interpret a predictive signal as biology?

# 9. Prepare ML-ready train and test sets

Return to the group-aware split.

In [ ]:
X_train = train_df[gene_cols].copy()
X_test = test_df[gene_cols].copy()

y_train = train_df["cancer"].copy()
y_test = test_df["cancer"].copy()

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

Check missingness before proceeding.

In [ ]:
print("Missing train values:",
      X_train.isna().sum().sum())

print("Missing test values:",
      X_test.isna().sum().sum())


# 11. Independent task - UCI Mice Protein Expression

Now apply the same general workflow to the UCI Mice Protein Expression dataset.

    Data_Cortex_Nuclear.xls

The dataset is available from the UCI Machine Learning Repository at:
https://archive.ics.uci.edu/dataset/342/mice%2Bprotein%2Bexpression

The dataset contains 1,080 rows, expression levels for 77 proteins or protein modifications, measurements from 72 mice, and 15 measurements per mouse. It also includes MouseID, Genotype, Treatment, Behavior, and an eight-level composite class.

### What are the 15 measurements?

For each mouse sample, the original reverse-phase protein array (RPPA) experiment used three technical replicate spots at each point of a five-point dilution series:

    1 mouse × 5 dilution points × 3 technical replicates = 15 rows

The dilution points are different concentrations of the same sample, and the replicate spots are repeated technical measurements. They are not 15 independent mice or 15 biological replicates. The public UCI spreadsheet does not contain separate dilution-level or replicate-number columns, so the exact dilution curve cannot be reconstructed  alone, meaning you need to select one dilution level. In the distributed file, labels such as `309_1` through `309_15` are row-level measurement identifiers: the part before the final underscore identifies the biological mouse, while the suffix identifies the measurement record.

Create a derived grouping variable before splitting:

    mice["mouse_id"] = mice["MouseID"].astype(str).str.rsplit("_", n=1).str[0]

Your goal is to turn the data into a defensible ML-ready dataset while keeping measurement, mouse, experimental-condition, and target roles separate.

## Required tasks

### A. Data presentation

Identify:

- what one row represents and what one biological mouse represents;
- the 77 numerical protein-expression measurements;
- MouseID, derived mouse_id, and the experimental metadata;
- one possible prediction target.

Consider Genotype as a primary target: control versus trisomy. You may choose the eight-level class target instead, but explain why. Remember that class is constructed from genotype, behavior, and treatment.

### B. EDA and tidy-data assessment

- load the Excel file and inspect dimensions, column names, and data types;
- inspect the distributions of Genotype, Treatment, Behavior, and class;
- check how many rows and how many unique mice belong to each group;
- visualize at least three biologically meaningful relationships.

Possible plots include:

- distributions of selected protein measurements by genotype;
- a PCA or two-dimensional projection colored by genotype, treatment, or behavior;
- protein-expression relationships or a correlation heatmap;
- the number of measurements per derived mouse_id.

For every plot, state the biological question it addresses.

### C. Diagnostics

Check:

- missingness in the protein measurements and metadata;
- duplicate rows and duplicate MouseID measurement records;
- whether every mouse has the expected number of repeated measurements;
- unusual categories or inconsistent labels;
- non-finite, negative, or otherwise implausible expression values;
- highly correlated or redundant protein variables;
- whether any candidate predictor is a deterministic part of the target.

### D. Preparation

Choose one supervised-learning problem and create:

    X
    y

Explain every variable that you exclude.

For a genotype-prediction question, start with the 77 protein measurements as candidate features. Exclude:

- MouseID and derived mouse_id, because the raw identifier identifies a measurement record and the derived identifier identifies the biological unit;
- the target column;
- class when Genotype is the target, because it contains genotype information;
- Treatment and Behavior unless you explicitly want a conditional prediction question and can justify their use.

Use the real missing values in the dataset to compare a defensible removal rule with an imputation rule. If you impute, fit the imputer on the training data only and apply it to the test data.

### E. Split

Create an appropriate training/test split.

For generalization to unseen mice:

- derive and group by mouse_id;
- keep all 15 measurements from a mouse in the same partition;
- use a group-aware split such as GroupShuffleSplit or StratifiedGroupKFold when appropriate.

Explain:

- what population you want the model to generalize to;
- why a row-wise random split would give an over-optimistic estimate;
- what the independent biological unit is;
- whether stratification is needed and whether it is compatible with your grouping strategy.

### F. Confounding and distribution structure

Investigate associations between the target and:

- treatment;
- behavior;
- genotype;
- the composite class;
- the number of measurements per mouse.

Use cross-tabulations and plots to ask:

- are treatment and behavior balanced across genotype?
- does the composite class encode the target directly?
- do some mice or experimental groups contribute different numbers of rows?
- could a model exploit experimental design structure instead of protein biology?

Identify at least one plausible source of confounding, target leakage, or distribution shift.

### G. Deliverable

Provide:

- X_train, X_test, y_train, y_test;
- a table documenting the role of each column;
- a missingness report and a justified removal or imputation decision;
- plots if you want;
- a short Markdown interpretation describing:
  - the ML question;
  - the biological unit;
  - the split strategy;
  - one potential source of bias, confounding, or leakage.

## Optional advanced task

Compare two evaluation questions:

1. a random row-wise train/test split;
2. a mouse-grouped train/test split.

Are these evaluating the same kind of generalization?

As a second extension, compare predicting Genotype with predicting the composite class. Which question is biologically clearer, and which variables must be excluded to avoid target leakage?


# 12. Take-home messages

- Biological data should be understood before they are modelled.
- High-dimensional omics commonly have \(p \gg n\).
- Metadata are part of the scientific problem, not administrative decoration.
- Data diagnostics include checking experimental structure, not only detecting `NaN`.
- Similar or repeated observations must remain in the same split.
- Data augmentation and pseudo-replication can produce leakage if derivatives cross the split boundary.
- Batch effects are technical signals that a model can exploit.
- Batch–target confounding makes biological interpretation especially difficult.
- Train/test splitting is part of experimental design.
- A valid ML pipeline begins with a valid scientific question.

# 13. Data-preparation readiness checklist

Use this checklist before starting any biological machine-learning analysis. Tick an item only after inspecting the data or documenting a justified decision. It is dataset-independent and can be reused in later practicals.

## Scientific purpose and provenance

- [ ] Is the biological question and prediction task stated clearly?
- [ ] Is the unit of observation clear (for example, patient, tissue, cell, mouse, or technical measurement)?
- [ ] Do I know where the data came from, how they were generated, and under which license or access conditions they may be used?
- [ ] Have I recorded the data version, processing history, and transformations already applied?

## Table shape and identifiers

- [ ] Is the table tidy: one observation per row, one variable per column, and one value per cell?
- [ ] If the original assay has features in rows and samples in columns, have I transposed it into the format expected by the analysis tool?
- [ ] Are row names, column names, and identifiers clear, unique, stable, and free from accidental index columns?
- [ ] Can every feature and observation be traced back to its source?
- [ ] Are the feature matrix and metadata aligned by an explicit identifier rather than by row order?

## Types, values, and measurement meaning

- [ ] Are numerical, categorical, ordinal, text, date, and identifier columns correctly classified?
- [ ] Are units, scales, encodings, reference categories, and detection limits understood?
- [ ] Have I checked impossible values, inconsistent labels, unexpected categories, constant features, and extreme values?
- [ ] Are normalization, scaling, log transformation, and batch-correction steps documented?

## Missing values

- [ ] Have I detected missing values using the dataset's actual missing-value representations, not only `NaN`?
- [ ] Have I measured missingness by row, column, group, and relevant batch or time variable?
- [ ] Do I know whether missingness is structural, technical, or plausibly related to the biology or target?
- [ ] Have I justified whether to retain, remove, or impute missing values?
- [ ] If imputing, will the imputer be fitted on training data only and then applied to validation/test data?

## Replicates, dependence, and grouping

- [ ] Have I checked for exact duplicates and near-duplicate observations?
- [ ] Are observations biological replicates, technical replicates, repeated measurements, pseudo-replicates, or independent samples?
- [ ] Is the independent biological unit identified by a grouping variable?
- [ ] Will all related observations and their derivatives remain in the same train/validation/test split?
- [ ] If I created augmented or synthetic observations, is their provenance recorded and reproducible?

## Target, leakage, and study design

- [ ] Is the target defined before modelling, with its classes or values checked for balance and validity?
- [ ] Have I excluded identifiers, post-outcome variables, duplicated target information, and other leakage sources from the features?
- [ ] Have I identified batch, site, plate, subject, cohort, treatment, sex, age, time, or other possible confounders?
- [ ] Have I checked whether the target is associated with a technical or grouping variable?
- [ ] Does the planned split represent the biological generalization question?

## Reproducibility and decision record

- [ ] Are random seeds, software versions, file names, and input/output shapes recorded?
- [ ] Are all preprocessing steps placed in a reproducible pipeline?
- [ ] Have I recorded unresolved data-quality concerns and the decisions made about them?
- [ ] Can another person explain what each row, column, feature, target, and grouping variable means?

## Short preparation summary

Before modelling, write down:

- biological unit: ...
- number of observations and features: ...
- target and feature definition: ...
- missing-value decision: ...
- replicate/grouping decision: ...
- main confounder or possible leakage source: ...
- train/test split and why it is appropriate: ...
- remaining concerns: ...